# SampleSift: select and export OBIS occurrences

Setup once: `task jupyter:setup`.
Run `task backend:up`, then `task jupyter:lab`.
The widget includes its frontend bundle. The notebook queries the OBIS backend directly.


In [ ]:
import json
from pathlib import Path

import geopandas as gpd

from samplesift.jupyter import SampleSift

widget = SampleSift(
    api_url="http://127.0.0.1:8002/samplesift",
    collection="occurrences",
    tile_layer="occurrences",
    query_state={
        "predicates": [{"field": "depth", "operator": ">=", "value": 50}],
        "sort": ["date"],
        "selection": None,
    },
)
widget


## Draw a search region

Wait for the table to load. The initial filter selects depth of at least 50 metres.
Use the map's Polygon tool to draw a region.
Wait for the table to update before running the next cell.
The state below contains the scalar filters, sorting, and selected polygon.


In [ ]:
widget.query_state


## Alternative: select the Viking shipping forecast area

Viking covers 0–4°E and 58°30′–61°N.
Its corners are published in the [Met Office Shipping Forecast factsheet, Table 1 (PDF page 12)](https://www.metoffice.gov.uk/api/assets/file/factsheet_8_shipping_forecast_2025pdf?prefix=assets).
The included shapefile and GeoJSON were generated from those published coordinates.
They use EPSG:4326. They are not downloaded official GIS products.

Run either example below to replace your drawn polygon with Viking.
Both examples preserve the depth filter and sorting.
See `regions/README.md` for the fixture details.


In [ ]:
# Load Viking from the shapefile.
viking = gpd.read_file(Path("regions/viking.shp")).to_crs("EPSG:4326")
widget.set_selection(viking.geometry.iloc[0])
viking


## Select the same region using GeoJSON

This sets the same map polygon and search region without a geometry library.
Run this cell as an alternative to the shapefile cell above.


In [ ]:
viking_geojson = json.loads(Path("regions/viking.geojson").read_text())
widget.set_selection(viking_geojson)
widget.query_state


Keep the shapefile's `.dbf`, `.shx`, `.prj`, and `.cpg` files beside its `.shp` file.
For your own shapefiles, reproject to EPSG:4326 before calling `set_selection()`.
The component currently supports one polygon without holes.
Check `widget.query_error` if a state cannot be applied.
Wait for the table to update before exporting.


## Link a depth slider

Release this ipywidgets slider to set the minimum depth in metres.
The observer preserves the polygon, sorting, and filters on other fields.
Wait for the table to update before exporting.


In [ ]:
import ipywidgets as widgets

depth_slider = widgets.IntSlider(value=50, min=0, max=200, step=5,
                                 description="Min depth", continuous_update=False)

def set_min_depth(change):
    predicates = [p for p in widget.query_state["predicates"] if p["field"] != "depth"]
    widget.query_state = {**widget.query_state, "predicates": predicates + [
        {"field": "depth", "operator": ">=", "value": change["new"]}
    ]}

depth_slider.observe(set_min_depth, names="value")
depth_slider


## Fetch every matching occurrence

Wait for the selected region's table to finish updating.
This fetches all pages using a fixed copy of the table's successful query.
The example uses 100 rows per request to demonstrate pagination within the smaller region.
The browser's page and map stay unchanged.
Records can change on the server during export; this is not a database snapshot.


In [ ]:
df = widget.to_dataframe(all_rows=True, page_size=100)
print(f"Fetched {len(df)} matching occurrences")
df.head()


## Export the DataFrame

This writes the exact DataFrame above, without another API request.
For a direct fresh export instead, use `widget.to_csv("selected-occurrences.csv", page_size=100)`.


In [ ]:
csv_path = Path("selected-occurrences.csv")
df.to_csv(csv_path, index=False)
csv_path
